# Chatbot Evaluation with LangSmith

This notebook demonstrates how to evaluate chatbot applications using LangSmith with LLM-as-a-Judge evaluators.

**Stack:**
- LLM: Groq (Llama 3.3 70B and Llama 3.1 8B for comparison)
- Evaluation Framework: LangSmith
- Evaluation Approach: LLM-as-a-Judge

**Evaluators:**
1. Correctness: Uses LLM to grade if answer matches reference
2. Concision: Checks if response length is reasonable (< 2x reference)

## Step 1: Environment Setup

In [21]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0
)


## Step 2: Create a Test Dataset in LangSmith

Set up evaluation dataset with test cases in LangSmith.

In [22]:
from langsmith import Client

client = Client()

dataset_name = "Chatbots Evaluation"

# Check if exists, create only if not
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name)
    client.create_examples(
        dataset_id=dataset.id,
        examples=[...] # your examples
    )
else:
    dataset = client.read_dataset(dataset_name=dataset_name)

print(f"Dataset ID: {dataset.id}")

Dataset ID: 23f92921-8279-47d8-811c-3a0c73d19837


## Step 3: Define Evaluation Metrics (LLM-as-a-Judge)

Implement correctness and concision evaluators using LLM for grading.

In [23]:
from langchain_groq import ChatGroq
from langsmith import wrappers

groq_client = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0
)

eval_instructions = (
    "You are an expert professor specialized in "
    "grading students' answers to questions."
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = groq_client.invoke([
        {"role": "system", "content": eval_instructions},
        {"role": "user", "content": user_content},
    ]).content

    return response.strip() == "CORRECT"

In [24]:
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(
        len(outputs["response"]) < 2 * len(reference_outputs["answer"])
    )

## Step 4: Define the Chatbot Application

Create a simple chatbot that responds to user questions with concise answers.

In [25]:
default_instructions = (
    "Respond to the users question in a short, "
    "concise manner (one short sentence)."
)

def my_app(
    question: str,
    model: str = "llama-3.3-70b-versatile",
    instructions: str = default_instructions,
) -> str:
    return groq_client.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ]).content

## Step 5: Run the Evaluation

Execute evaluation on the Groq Llama 3.3 70B model.

In [26]:
# Wrapper function that maps dataset inputs to app outputs
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

# Evaluate with Groq Llama
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="groq-llama-70b-chatbot",
)

View the evaluation results for experiment: 'groq-llama-70b-chatbot-21bec33a' at:
https://smith.langchain.com/o/6d419d79-1f69-49f1-9409-d187df487fdf/datasets/23f92921-8279-47d8-811c-3a0c73d19837/compare?selectedSessions=1c492a68-fc2c-45b4-88c1-e08551bfa890




0it [00:00, ?it/s]

## Step 6: Compare Models

Compare performance between Llama 3.3 70B (larger) and Llama 3.1 8B (smaller, faster) models to evaluate quality vs speed tradeoffs.

In [27]:
# Wrapper for Groq Llama 3.3 8b (smaller/faster model for comparison)
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"], model="llama-3.1-8b-instant")}

# Evaluate with smaller model
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="groq-llama-8b-chatbot",
)

View the evaluation results for experiment: 'groq-llama-8b-chatbot-c1f10504' at:
https://smith.langchain.com/o/6d419d79-1f69-49f1-9409-d187df487fdf/datasets/23f92921-8279-47d8-811c-3a0c73d19837/compare?selectedSessions=32ea84a0-6b63-47c0-8c65-6e66f7bd9041




0it [00:00, ?it/s]